# Lab 03: Amazon Lambda (boto3)

Builds a function that makes a thumbnail whenever an image lands in an S3 bucket.
Run `make start LAB=03-lambda` first to create the VPC, security group and IAM role the lab gives you.

## Setup

Settings come from `.env`. Checking target to know wher ecalls go.

In [ ]:
import os
import io
import uuid

from PIL import Image, ImageDraw

import boto3
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)
print("Target:", os.environ.get("AWS_ENDPOINT_URL", "real AWS"))

REGION = boto3.session.Session().region_name
FUNCTION = "Create-Thumbnail"
ROLE = "lambda-execution-role"

iam = boto3.client("iam")
ec2 = boto3.client("ec2")
lambda_ = boto3.client("lambda")
s3 = boto3.client("s3")

## Starting state

`make start` created these. we find them by their attributes, not by IDs.

In [ ]:
role_arn = iam.get_role(RoleName=ROLE)['Role']['Arn']

vpc_id = next(
    v['VpcId']
    for v in ec2.describe_vpcs()['Vpcs']
    if v['CidrBlock']=='10.0.0.0/16'
)

subnet_id = next(
    sn['SubnetId']
    for sn in ec2.describe_subnets()['Subnets']
    if sn['VpcId']==vpc_id and sn['CidrBlock']=='10.0.1.0/24'
)

group_id = next(
    g['GroupId']
    for g in ec2.describe_security_groups()['SecurityGroups']
    if g['VpcId']==vpc_id and 'LambdaSecurityGroup' in g['GroupName']
)

print(role_arn, vpc_id, subnet_id, group_id, sep="\n")

Remove leftover `Create-Thumbnail` from previous run:

In [ ]:
try:
    lambda_.delete_function(FunctionName=FUNCTION)
except lambda_.exceptions.ResourceNotFoundException:
    pass

## Task 1: Create the Amazon S3 buckets 

Bucket names are global on AWS, so a random suffix used to keep them unqiue.
Thumbnails are written to the second bucket `OUTPUT_BUCKET`.

In [ ]:
suffix = uuid.uuid4().hex[:10]
SOURCE_BUCKET = f"images-{suffix}"
OUTPUT_BUCKET = f"{SOURCE_BUCKET}-resized"

for bucket_name in (SOURCE_BUCKET, OUTPUT_BUCKET):
    if REGION == 'us-east-1':
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': REGION}
        )

print(f"Input bucket:  {SOURCE_BUCKET}")
print(f"Output bucket: {OUTPUT_BUCKET}")

Here we're drawing our 1280x853 friend smiley.

In [ ]:
def make_smiley(width, height):
    """Draw smiley, def not a Serani Poji reference"""
    bg_color = 100, 100, 220
    image = Image.new("RGB", (width, height), bg_color)
    draw = ImageDraw.Draw(image)

    cx, cy = width/2, height/2
    r = min(width, height) * 0.35
    line_w = max(2, int(r*0.08))
    ink = 0, 0, 0

    # Head
    draw.ellipse(
        (cx-r, cy-r, cx+r, cy+r),
        fill=(255,220,0), outline=ink, width=line_w
    )

    # Eyes
    eye_r  = r*0.01
    eye_dx = r*0.03
    eye_y  = cy - r*0.2
    for side in (-1, 1):
        ex = cx + side*eye_dx
        draw.ellipse(
            (ex-eye_r, eye_y-eye_r, ex+eye_r, eye_y+eye_r),
            fill=ink
        )

    # Smile
    draw.arc(
        (cx-r*0.6, cy-r*0.6, cx+r*0.6, cy+r*0.6),
        start=30, end=150, fill=ink, width=line_w
    )

    out = io.BytesIO()
    image.save(out, format="JPEG")
    return out.getvalue()

s3.put_object(
    Bucket=SOURCE_BUCKET, Key='face.jpg',
    Body=make_smiley(1280,853)
)

## Task 2: Create an AWS Lambda function

## Task 3: Test your function

In [ ]:
# One cell per lab step, use assertion on result

## Task 4: Monitoring and Logging

In [ ]:
# One cell per lab step, use assertion on result

## Clean up

Remove everything this notebook created so `make end` can destroy the seed without dependency errors.

In [ ]:
# Delete what the tasks above created